**Importer les libraries pertinentes**

Source : Prettenhofer, P., Grisel, O., Blondel, M., Amor, A. et Buitinck, L. [Classification of text documents using sparse features](https://scikit-learn.org/stable/auto_examples/text/plot_document_classification_20newsgroups.html#sphx-glr-auto-examples-text-plot-document-classification-20newsgroups-py). *scikit-learn*. 

In [ ]:
import pandas as pd
import os
import re
import multiprocessing
cores = multiprocessing.cpu_count()
from collections import Counter
from IPython.display import clear_output

# Stopwords & punctuation
import string
from nltk.corpus import stopwords
punct = string.punctuation
stopw = stopwords.words('english') + [x for x in punct]
stopw += [x.translate
    (str.maketrans('', '', punct)) for x in stopwords.words('english')]

stopw +=  ["'d", "'ll", "'re", "'s", "'ve", '``', 'could', 'might', 'must', "n't", 'need', 'sha', 'wo', 'would']

# Tokenizer
from nltk.tokenize import word_tokenize

# Vectorizer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# Supervised learning
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Naive Bayes (GaussianNB)
from sklearn.naive_bayes import ComplementNB # "Particularly suited for imbalanced data sets" (sklearn documentation)

# K plus proches voisins (kNN)
from sklearn.neighbors import KNeighborsClassifier

# Machines à vecteurs de support (SVM)
from sklearn.svm import LinearSVC

# Régression Logistique
from sklearn.linear_model import LogisticRegression

# Forêt aléatoire 
from sklearn.ensemble import RandomForestClassifier

# Perceptron multicouches
from sklearn.neural_network import MLPClassifier

# Validation croisée - 5-fold cross-validation
stratified_kfold = StratifiedKFold(shuffle=True, random_state=42)

**Charger les données nettoyées**

In [ ]:
folder = '../2-scripts/1-preprocessing'
datasets = os.listdir(folder)
datasets

**Entraîner les modèles et sortir les résultats**

In [ ]:
def tokenize_remove_stopwords(text):
 for token in word_tokenize(text):
    if token in stopw: continue
    yield (token)

In [ ]:
for dataset in datasets[5:]:
    ratio = int(dataset[22:-7])
    df = pd.read_excel(os.path.join(folder, dataset))
    report = []    

    corpus = df['cleaned_text_post'].astype('str')
    y = df['category'].astype('str')

    # Valeurs à tester pour le nombre de traits discriminants
    n_features_values = [100, 250, 500, 750, 1000, 2500, 5000, 10000, 15000]

    # Algorithmes à tester
    algorithms = {
        # 'Complement Naive Bayes' : ComplementNB(), 
        # 'kNN (k=1)' : KNeighborsClassifier(n_neighbors=1),
        # 'kNN (k=2)' : KNeighborsClassifier(n_neighbors=2),
        # 'kNN (k=3)' : KNeighborsClassifier(n_neighbors=3),
        # 'Support Vector Machines (SVM)' : LinearSVC(dual="auto"),
        # 'Logistic Regression': LogisticRegression(n_jobs = cores),
        'Random Forest': RandomForestClassifier(n_jobs = cores),
        # 'Multilayered Perceptron': MLPClassifier(max_iter=500)
    }

    for n_features in n_features_values:
            tf_idf_vectorizer = TfidfVectorizer(
                 stop_words = stopw,
                 tokenizer = word_tokenize,
                 token_pattern = None,
                 max_features = n_features,
            )            
            X = tf_idf_vectorizer.fit_transform(corpus).toarray()

            for algorithm in algorithms.keys():
                # Validation croisée 
                predictions = cross_val_predict(algorithms[algorithm], X, y, cv=stratified_kfold)
                result = classification_report(y, predictions, output_dict=True)

                # Performances (rappel, précision, score F)
                results = {
                    '% Incels' : int(ratio),
                    'Algorithme' : algorithm,
                    'Modèle de vectorisation': 'TF-IDF',
                    'N traits discr.' : n_features,
                    'accuracy': result['accuracy']    
                }

                results.update(result['macro avg'])
                report.append(results)
                
                print(algorithm, f' | {ratio}% Incels | ', f'{n_features} traits discr.\n', classification_report(y, predictions))
                clear_output(wait=True)

    # Exporter les résults
    # support = Nombre d'occurrence dans chaque classe
    report = pd.DataFrame(report)
    report['nb_posts_incels'] = (report['% Incels'].apply(lambda x: x/100) * report['support']).astype(int)
    report['nb_posts_neutral'] = (report['% Incels'].apply(lambda x: 1-(x/100)) * report['support']).astype(int)

    report = report.rename(columns={'support':'nb_posts_total'})
    report['nb_posts_total'] = report['nb_posts_total'].astype(int)

    # Réorganiser l'ordre des colonnes
    report = report[['nb_posts_total', 'nb_posts_incels', 'nb_posts_neutral', '% Incels', 'Algorithme', 
                    'Modèle de vectorisation', 'N traits discr.', 'accuracy', 'precision', 'recall','f1-score']]

    report.sort_values(by='f1-score', ascending=False).to_csv(
    f'../3-results/results_training_tfidf_{ratio}pc_RandomForest_MultiLayeredPerceptron.csv', index=False
    )
    

**Tester le système retenu sur de nouvelles données**

*On reproduit l'apprentissage pour les systèmes retenus uniquement, puis on l'applique aux données test (inédites)*

In [ ]:
folder = '../2-scripts/1-preprocessing'
datasets_training = os.listdir(folder)[3:6]
datasets_training

*Données test*

In [ ]:
file_test = '../1-data/test_dataset_10pc.xlsx'

df_test = pd.read_excel(file_test)
df_test

*Valider que les données test n'ont jamais été vues*

*Fichier 40% données incels*

In [ ]:
df_training = pd.read_excel('../1-data/training_datasets/train_dataset_40pc.xlsx')

validate = pd.concat([df_training, df_test])
validate[validate.duplicated(subset='id_post')]

*Fichier 50% données incels*

In [ ]:
df_training = pd.read_excel('../1-data/training_datasets/train_dataset_50pc.xlsx')

validate = pd.concat([df_training, df_test])
validate[validate.duplicated(subset='id_post')]

*Fichier 60% données incels*

In [ ]:
df_training = pd.read_excel('../1-data/training_datasets/train_dataset_60pc.xlsx')

validate = pd.concat([df_training, df_test])
validate[validate.duplicated(subset='id_post')]

In [ ]:
# Système retenus - top 10
## 40, 50, 60 % incels
## 5000, 10 000, 15 000 traits discriminants
## Complement Naive Bayes, Support Vector Machines, Logistic regression

# On prend le jeu de données test nettoyées 
df_test = pd.read_excel('../1-data/cleaned_test_dataset_10pc.xlsx')
corpus_test = df_test['cleaned_text_post'].astype('str')
y_test = df_test['category'].astype('str')

report = [] 
for dataset in datasets_training:
    ratio = int(dataset[22:-7])

    df_train = pd.read_excel(os.path.join(folder, dataset))  
    corpus_train  = df_train['cleaned_text_post'].astype('str') 
    y_train = df_train['category'].astype('str') 

    # Valeurs à tester pour le nombre de traits discriminants
    n_features_values = [5000, 10000, 15000]

    # Algorithmes à tester
    algorithms = {
        'Complement Naive Bayes' : ComplementNB(), 
        'Support Vector Machines (SVM)' : LinearSVC(dual="auto"),
        'Logistic Regression': LogisticRegression(n_jobs = cores),
    }

    for n_features in n_features_values:
            tf_idf_vectorizer = TfidfVectorizer(
                 stop_words = stopw,
                 tokenizer = word_tokenize,
                 token_pattern = None,
                 max_features = n_features,
            )            
            
            X_train = tf_idf_vectorizer.fit_transform(corpus_train).toarray()
            X_test = tf_idf_vectorizer.fit_transform(corpus_test).toarray()

            for algorithm in algorithms.keys():
                model = algorithms[algorithm]

                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)

                # Calculer les performances
                result = classification_report(y_test, y_pred, output_dict=True)
                results = {
                    '% Incels' : int(ratio),
                    'Algorithme' : algorithm,
                    'Modèle de vectorisation': 'TF-IDF',
                    'N traits discr.' : n_features,
                    'accuracy': result['accuracy']    
                }

                results.update(result['macro avg'])
                report.append(results)
                
                print(algorithm, f' | {ratio}% Incels | ', f'{n_features} traits discr.\n', classification_report(y_test, y_pred))
                clear_output(wait=True)

# Exporter les résults
# support = Nombre d'occurrence dans chaque classe
report = pd.DataFrame(report)
report['nb_posts_incels'] = (report['% Incels'].apply(lambda x: x/100) * report['support']).astype(int)
report['nb_posts_neutral'] = (report['% Incels'].apply(lambda x: 1-(x/100)) * report['support']).astype(int)

report = report.rename(columns={'support':'nb_posts_total'})
report['nb_posts_total'] = report['nb_posts_total'].astype(int)

# Réorganiser l'ordre des colonnes
report = report[['nb_posts_total', 'nb_posts_incels', 'nb_posts_neutral', '% Incels', 'Algorithme', 
                'Modèle de vectorisation', 'N traits discr.', 'accuracy', 'precision', 'recall','f1-score']]

report.sort_values(by='f1-score', ascending=False).to_csv(
f'../3-results/results_test_top_10.csv', index=False
)